In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to create summaries
def summarize_df(df, filename, name_col=None):
    rows = df.shape[0]
    cmdb_unique = df['cmdb_id'].dropna().unique()
    cmdb_count = len(cmdb_unique)
    cmdb_sample = pd.Series(cmdb_unique).head(50)
    # name_col: metric->kpi_name, trace->trace_name, log->log_name; for error logs fallback to 'message' if present
    if name_col and name_col in df.columns:
        names_unique = df[name_col].dropna().unique()
        names_sample = pd.Series(names_unique).head(50)
    elif 'message' in df.columns:
        names_unique = df['message'].dropna().unique()
        names_sample = pd.Series(names_unique).head(50)
        name_col = 'message'
    else:
        names_sample = pd.Series([], dtype=object).head(0)
        name_col = None
    # timestamps
    if 'timestamp' in df.columns:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        ts_min = ts.min()
        ts_max = ts.max()
        ts_min_s = ts_min.strftime('%Y-%m-%d %H:%M:%S %Z%z') if pd.notnull(ts_min) else None
        ts_max_s = ts_max.strftime('%Y-%m-%d %H:%M:%S %Z%z') if pd.notnull(ts_max) else None
    else:
        ts_min_s = ts_max_s = None
    summary = pd.DataFrame({
        'file': [filename],
        'rows': [rows],
        'unique_cmdb_count': [cmdb_count],
        'min_timestamp': [ts_min_s],
        'max_timestamp': [ts_max_s],
        'name_column': [name_col if name_col is not None else None]
    }).set_index('file')
    return summary, cmdb_sample, names_sample

# Summaries
summary_metric, metric_cmdb_ids, metric_kpis = summarize_df(df_metric, 'metric.csv', name_col='kpi_name')
summary_trace, trace_cmdb_ids, trace_names = summarize_df(df_trace, 'trace.csv', name_col='trace_name')
summary_log, log_cmdb_ids, log_names = summarize_df(df_log, 'log.csv', name_col='log_name')
summary_error, error_cmdb_ids, error_msgs = summarize_df(df_error, 'error_logs.csv', name_col=None)

# Compact top-20 rows for each
metric_top20 = df_metric.head(20)
trace_top20 = df_trace.head(20)
log_top20 = df_log.head(20)
error_top20 = df_error.head(20)

# Display compact results (variables; IPython will render them)
summary_metric, metric_cmdb_ids, metric_kpis, metric_top20, \
summary_trace, trace_cmdb_ids, trace_names, trace_top20, \
summary_log, log_cmdb_ids, log_names, log_top20, \
summary_error, error_cmdb_ids, error_msgs, error_top20

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv:\n"
    "- Total rows: 322,448\n"
    "- Unique cmdb_id count: 59 (examples include adservice, adservice-0, adservice-1, ..., recommendationservice-1)\n"
    "- KPI names: many distinct kpi_name values (sample contains runtime and app metrics such as app.grpc.count, app.http.mrt, runtime.java_lang_...)\n"
    "- Timestamp range (UTC+8): 2022-03-21 09:30:00 to 2022-03-21 10:30:00\n"
    "- Top 20 rows were displayed in the prior execution for inspection.\n\n"
    "trace.csv:\n"
    "- Total rows: 35,512\n"
    "- Unique cmdb_id count: 40 (examples include adservice-0, adservice-1, adservice-2, cartservice-0, shippingservice-2)\n"
    "- Trace names: many trace_name values (examples include trace.from_frontend-0.duration_mean, trace.from_frontend-0.error_rate, trace.to_cartservice-2.duration_mean)\n"
    "- Timestamp range (UTC+8): 2022-03-21 10:00:00 to 2022-03-21 10:29:00\n"
    "- Top 20 rows were displayed in the prior execution for inspection.\n\n"
    "log.csv:\n"
    "- Total rows: 1,830\n"
    "- Unique cmdb_id count: 31 (examples include adservice-0, cartservice-0, checkoutservice-0, redis-cart-0)\n"
    "- Log names: log.error_count and log.row_count\n"
    "- Timestamp range (UTC+8): 2022-03-21 10:00:00 to 2022-03-21 10:30:00\n"
    "- Top 20 rows were displayed in the prior execution for inspection.\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (empty)\n"
    "- Unique cmdb_id count: 0\n"
    "- No messages or timestamps present.\n\n"
    "Overall: metric.csv is the largest dataset spanning 09:30–10:30, trace and log data cover 10:00–10:30 window, and there are no error log records. "
    "You can proceed with targeted analysis (filtering or anomaly detection) now that the basic inventories and time ranges are known."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              rows  unique_cmdb_count                 min_timestamp                 max_timestamp name_column
file                                                                                                         
metric.csv  322448                 59  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    kpi_name, 0                    adservice
1                  adservice-0
2                  adservice-1
3                  adservice-2
4                   adservice2
                ...           
45     productcatalogservice-2
46    productcatalogservice2-0
47       recommendationservice
48     recommendationservice-0
49     recommendationservice-1
Length: 50, dtype: object, 0                                        app.grpc.count
1                                          app.grpc.mrt
2                                           app.grpc.rr
3                                           app.grpc.sr
4                                        app.http.count
                            ...                        
45    runtime.java_lang_GarbageCollector_LastGcInfo_...
46    runtime.java_lang_GarbageCollector_LastGcInfo_...
47    runtime.java_lang_GarbageCollector_LastGcInfo_...
48    runtime.java_lang_GarbageCollector_LastGcInfo_...
49    runtime.java_lang_GarbageCollector_LastGcInfo_...
Length: 50, dtype: object,      timestamp    cmdb_id                                           kpi_name         value
0   1647826200  adservice                                     app.grpc.count     88.000000
1   1647826200  adservice                                       app.grpc.mrt      0.002927
2   1647826200  adservice                                        app.grpc.rr    100.000000
3   1647826200  adservice                                        app.grpc.sr    100.000000
4   1647826200  adservice                                     app.http.count     24.000000
5   1647826200  adservice                                       app.http.mrt      0.009923
6   1647826200  adservice                                        app.http.rr    100.000000
7   1647826200  adservice                                        app.http.sr    100.000000
8   1647826200  adservice    runtime.java_lang_ClassLoading_LoadedClassCount   5162.000000
9   1647826200  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...   5185.000000
10  1647826200  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     23.000000
11  1647826200  adservice             runtime.java_lang_ClassLoading_Verbose      0.000000
12  1647826200  adservice  runtime.java_lang_Compilation_CompilationTimeM...      1.000000
13  1647826200  adservice  runtime.java_lang_Compilation_TotalCompilation...  47021.000000
14  1647826200  adservice  runtime.java_lang_GarbageCollector_CollectionC...  38716.666667
15  1647826200  adservice  runtime.java_lang_GarbageCollector_CollectionC...      8.000000
16  1647826200  adservice  runtime.java_lang_GarbageCollector_CollectionT...  70281.750000
17  1647826200  adservice  runtime.java_lang_GarbageCollector_CollectionT...    480.000000
18  1647826200  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000
19  1647826200  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      1.000000,             rows  unique_cmdb_count                 min_timestamp                 max_timestamp name_column
file                                                                                                       
trace.csv  35512                 40  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace_name, 0                  adservice-0
1                  adservice-1
2                  adservice-2
3                 adservice2-0
4                cartservice-0
                ...           
35    recommendationservice2-0
36           shippingservice-0
37           shippingservice-1
38           shippingservice-2
39          shippingservice2-0
Length: 40, dtype: object, 0      trace.from_frontend-0.duration_mean
1       trace.from_frontend-0.duration_p95
2         trace.from_frontend-0.error_rate
3          trace.from_frontend-0.row_count
4      trace.from_frontend-1.duration_mean
                      ...                 
45     trace.to_cartservice-1.duration_p95
46       trace.to_cartservice-1.error_rate
47        trace.to_cartservice-1.row_count
48    trace.to_cartservice-2.duration_mean
49     trace.to_cartservice-2.duration_p95
Length: 50, dtype: object,      timestamp      cmdb_id                           trace_name      value
0   1647828000  adservice-0  trace.from_frontend-0.duration_mean   0.000022
1   1647828000  adservice-0   trace.from_frontend-0.duration_p95   0.000034
2   1647828000  adservice-0     trace.from_frontend-0.error_rate   0.000000
3   1647828000  adservice-0      trace.from_frontend-0.row_count  17.000000
4   1647828000  adservice-0  trace.from_frontend-1.duration_mean   0.000018
5   1647828000  adservice-0   trace.from_frontend-1.duration_p95   0.000026
6   1647828000  adservice-0     trace.from_frontend-1.error_rate   0.000000
7   1647828000  adservice-0      trace.from_frontend-1.row_count  17.000000
8   1647828000  adservice-0  trace.from_frontend-2.duration_mean   0.000018
9   1647828000  adservice-0   trace.from_frontend-2.duration_p95   0.000023
10  1647828000  adservice-0     trace.from_frontend-2.error_rate   0.000000
11  1647828000  adservice-0      trace.from_frontend-2.row_count  12.000000
12  1647828000  adservice-1  trace.from_frontend-0.duration_mean   0.000017
13  1647828000  adservice-1   trace.from_frontend-0.duration_p95   0.000023
14  1647828000  adservice-1     trace.from_frontend-0.error_rate   0.000000
15  1647828000  adservice-1      trace.from_frontend-0.row_count  18.000000
16  1647828000  adservice-1  trace.from_frontend-1.duration_mean   0.000017
17  1647828000  adservice-1   trace.from_frontend-1.duration_p95   0.000023
18  1647828000  adservice-1     trace.from_frontend-1.error_rate   0.000000
19  1647828000  adservice-1      trace.from_frontend-1.row_count  16.000000,          rows  unique_cmdb_count                 min_timestamp                 max_timestamp name_column
file                                                                                                    
log.csv  1830                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800    log_name, 0                 adservice-0
1                 adservice-1
2                 adservice-2
3               cartservice-0
4               cartservice-1
               ...           
26    recommendationservice-2
27          shippingservice-0
28          shippingservice-1
29          shippingservice-2
30               redis-cart-0
Length: 31, dtype: object, 0    log.error_count
1      log.row_count
dtype: object,      timestamp            cmdb_id         log_name  value
0   1647828000        adservice-0  log.error_count    0.0
1   1647828000        adservice-0    log.row_count  104.0
2   1647828000        adservice-1  log.error_count    0.0
3   1647828000        adservice-1    log.row_count   92.0
4   1647828000        adservice-2  log.error_count    0.0
5   1647828000        adservice-2    log.row_count   88.0
6   1647828000      cartservice-0  log.error_count    0.0
7   1647828000      cartservice-0    log.row_count  700.0
8   1647828000      cartservice-1  log.error_count    0.0
9   1647828000      cartservice-1    log.row_count  693.0
10  1647828000      cartservice-2  log.error_count    0.0
11  1647828000      cartservice-2    log.row_count  680.0
12  1647828000  checkoutservice-0  log.error_count    0.0
13  1647828000  checkoutservice-0    log.row_count   37.0
14  1647828000  checkoutservice-1  log.error_count    0.0
15  1647828000  checkoutservice-1    log.row_count   53.0
16  1647828000  checkoutservice-2  log.error_count    0.0
17  1647828000  checkoutservice-2    log.row_count   49.0
18  1647828000  currencyservice-0  log.error_count    0.0
19  1647828000  currencyservice-0    log.row_count  626.0,                 rows  unique_cmdb_count min_timestamp max_timestamp name_column
file                                                                           
error_logs.csv     0                  0          None          None     message, Series([], dtype: object), Series([], dtype: object), Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing loaded dataframes if present; otherwise load
try:
    df_metric
    df_trace
    df_log
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_trace = pd.read_csv('trace.csv')
    df_log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Convert timestamps to timezone-aware datetimes (UTC->Asia/Shanghai)
for df in [('df_metric', df_metric), ('df_trace', df_trace), ('df_log', df_log)]:
    name, d = df
    if 'ts_dt' not in d.columns:
        d['ts_dt'] = pd.to_datetime(d['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define window (inclusive) in UTC+8
window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)

# Helper to aggregate a dataframe grouped by two keys and compute required stats
def aggregate_group(df, key_cols, value_col, name_for_value):
    # mark in-window using ts_dt
    mask_col = 'in_window'
    df[mask_col] = df['ts_dt'].between(window_start, window_end)
    g = df.groupby(key_cols)
    agg = g.agg(
        total_point_count=(value_col, 'size'),
        min_timestamp=('ts_dt', 'min'),
        max_timestamp=('ts_dt', 'max'),
        p95=(value_col, lambda x: x.quantile(0.95)),
        p5=(value_col, lambda x: x.quantile(0.05)),
        count_in_window=(mask_col, 'sum')
    ).reset_index()
    # format timestamps compactly as strings in UTC+8
    agg['min_timestamp'] = agg['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    agg['max_timestamp'] = agg['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    # ensure numeric columns are real numbers
    agg['p95'] = agg['p95'].astype(float)
    agg['p5'] = agg['p5'].astype(float)
    # sort by count_in_window desc, then total_point_count desc
    agg = agg.sort_values(['count_in_window', 'total_point_count'], ascending=[False, False]).reset_index(drop=True)
    return agg

# 1) metric.csv aggregation by (cmdb_id, kpi_name)
metric_agg_full = aggregate_group(df_metric, ['cmdb_id', 'kpi_name'], 'value', 'kpi_name')
metric_agg_top20 = metric_agg_full.head(20)

# 2) trace.csv aggregation by (cmdb_id, trace_name)
trace_agg_full = aggregate_group(df_trace, ['cmdb_id', 'trace_name'], 'value', 'trace_name')
trace_agg_top20 = trace_agg_full.head(20)

# 3) log.csv aggregation by (cmdb_id, log_name)
log_agg_full = aggregate_group(df_log, ['cmdb_id', 'log_name'], 'value', 'log_name')
# return all groups (or top20 if many)
log_agg_top = log_agg_full.head(20)

# Display compact outputs (variables)
metric_agg_top20, trace_agg_top20, log_agg_top

```
Out[2]:
```
```python
summary = (
    "Aggregate summary (global percentiles computed on full series; window = 2022-03-21 10:00:00 to 2022-03-21 10:30:00 UTC+8):\n\n"
    "metric.csv (grouped by cmdb_id, kpi_name):\n"
    "- Top groups by points in window are all from 'adservice' KPIs. Many adservice KPIs have count_in_window = 31 and total_point_count = 61.\n"
    "- Example metrics: app.grpc.count (p95=193, p5=104), app.grpc.mrt (p95≈0.00297, p5≈0.00261), app.http.count (p95=24, p5=24).\n"
    "- Full metric series for these groups span 2022-03-21 09:30:00 to 2022-03-21 10:30:00 (UTC+8).\n\n"
    "trace.csv (grouped by cmdb_id, trace_name):\n"
    "- Top trace groups are from adservice-0 and adservice-1 (e.g., trace.from_frontend-*), each with count_in_window = 30 and total_point_count = 30.\n"
    "- Example trace stats: trace.from_frontend-0.duration_mean (p95≈0.000023, p5≈0.000017), row_count (p95≈23, p5≈14.9).\n"
    "- Trace timestamps for these groups span 2022-03-21 10:00:00 to 2022-03-21 10:29:00 (UTC+8).\n\n"
    "log.csv (grouped by cmdb_id, log_name):\n"
    "- Many log groups have count_in_window = 31 (e.g., adservice-1 log.error_count/log.row_count, cartservice-2, currencyservice-1/2, frontend-1, productcatalogservice-2, recommendationservice-1).\n"
    "- Example log stats: adservice-1 log.row_count (p95=114, p5=81); cartservice-2 log.row_count (p95=884, p5=696).\n"
    "- Log timestamps mostly span 2022-03-21 10:00:00 to 2022-03-21 10:30:00 (UTC+8).\n\n"
    "Notes:\n"
    "- error_logs.csv contains no rows.\n"
    "- Outputs shown are compact group summaries (top groups by count_in_window). Full group tables are stored in kernel variables for further analysis.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  total_point_count                 min_timestamp                 max_timestamp           p95            p5  count_in_window
0   adservice                                     app.grpc.count                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    193.000000    104.000000               31
1   adservice                                       app.grpc.mrt                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      0.002973      0.002605               31
2   adservice                                        app.grpc.rr                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    100.000000    100.000000               31
3   adservice                                        app.grpc.sr                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    100.000000    100.000000               31
4   adservice                                     app.http.count                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800     24.000000     24.000000               31
5   adservice                                       app.http.mrt                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      0.009923      0.009408               31
6   adservice                                        app.http.rr                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    100.000000    100.000000               31
7   adservice                                        app.http.sr                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    100.000000    100.000000               31
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800   5162.000000   5162.000000               31
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800   5185.000000   5185.000000               31
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800     23.000000     23.000000               31
11  adservice             runtime.java_lang_ClassLoading_Verbose                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      0.000000      0.000000               31
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      1.000000      1.000000               31
13  adservice  runtime.java_lang_Compilation_TotalCompilation...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800  47021.000000  47021.000000               31
14  adservice  runtime.java_lang_GarbageCollector_CollectionC...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800  38870.916667  38724.833333               31
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      8.000000      8.000000               31
16  adservice  runtime.java_lang_GarbageCollector_CollectionT...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800  70556.833333  70295.666667               31
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800    480.000000    480.000000               31
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      1.000000      1.000000               31
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                 61  2022-03-21 09:30:00 CST+0800  2022-03-21 10:30:00 CST+0800      1.000000      1.000000               31,         cmdb_id                           trace_name  total_point_count                 min_timestamp                 max_timestamp        p95         p5  count_in_window
0   adservice-0  trace.from_frontend-0.duration_mean                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000023   0.000017               30
1   adservice-0   trace.from_frontend-0.duration_p95                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000038   0.000024               30
2   adservice-0     trace.from_frontend-0.error_rate                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000000   0.000000               30
3   adservice-0      trace.from_frontend-0.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  23.000000  14.900000               30
4   adservice-0  trace.from_frontend-1.duration_mean                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000024   0.000017               30
5   adservice-0   trace.from_frontend-1.duration_p95                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000043   0.000024               30
6   adservice-0     trace.from_frontend-1.error_rate                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000000   0.000000               30
7   adservice-0      trace.from_frontend-1.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  23.000000  13.450000               30
8   adservice-0  trace.from_frontend-2.duration_mean                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000025   0.000016               30
9   adservice-0   trace.from_frontend-2.duration_p95                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000041   0.000020               30
10  adservice-0     trace.from_frontend-2.error_rate                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000000   0.000000               30
11  adservice-0      trace.from_frontend-2.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  16.550000   8.000000               30
12  adservice-1  trace.from_frontend-0.duration_mean                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000022   0.000016               30
13  adservice-1   trace.from_frontend-0.duration_p95                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000036   0.000017               30
14  adservice-1     trace.from_frontend-0.error_rate                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000000   0.000000               30
15  adservice-1      trace.from_frontend-0.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  22.550000  15.450000               30
16  adservice-1  trace.from_frontend-1.duration_mean                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000020   0.000016               30
17  adservice-1   trace.from_frontend-1.duration_p95                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000033   0.000018               30
18  adservice-1     trace.from_frontend-1.error_rate                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   0.000000   0.000000               30
19  adservice-1      trace.from_frontend-1.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  23.550000  14.000000               30,                     cmdb_id         log_name  total_point_count                 min_timestamp                 max_timestamp      p95      p5  count_in_window
0               adservice-1  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
1               adservice-1    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800   114.00    81.0               31
2             cartservice-2  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
3             cartservice-2    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800   884.00   696.0               31
4         currencyservice-1  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
5         currencyservice-1    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800   799.50   561.0               31
6         currencyservice-2  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
7         currencyservice-2    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800   798.50   569.5               31
8                frontend-1  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
9                frontend-1    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800  1616.50  1191.5               31
10  productcatalogservice-2  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
11  productcatalogservice-2    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800  1005.00   774.5               31
12  recommendationservice-1  log.error_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800     0.00     0.0               31
13  recommendationservice-1    log.row_count                 31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800   213.00   169.5               31
14              adservice-0  log.error_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800     0.00     0.0               30
15              adservice-0    log.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   127.30    93.8               30
16              adservice-2  log.error_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800     0.00     0.0               30
17              adservice-2    log.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   114.00    82.0               30
18            cartservice-0  log.error_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800     0.00     0.0               30
19            cartservice-0    log.row_count                 30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800   885.95   703.6               30)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# reuse existing variables: df_metric, metric_agg_full, window_start, window_end, tz
# Ensure required variables exist
try:
    df_metric
    metric_agg_full
    window_start
    window_end
    tz
except NameError:
    # load if missing (fallback)
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_agg_full = None
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)

# Filter metric rows within incident window (UTC+8)
df_metric_window = df_metric[df_metric['ts_dt'].between(window_start, window_end)].copy()

# Merge global thresholds (p95, p5) from previously computed metric_agg_full
# metric_agg_full should contain columns: cmdb_id, kpi_name, p95, p5
thresholds = metric_agg_full[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()
df_w = df_metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies based on global thresholds (do not recompute thresholds here)
eps = 1e-9
df_w['is_anomaly'] = (df_w['value'] >= df_w['p95']) | (df_w['value'] <= df_w['p5'])

# Compute per-row violation ratio:
# if high-side anomaly: abs(value - p95)/max(p95, eps)
# if low-side anomaly:  abs(value - p5)/max(p5, eps)
high_diff = (df_w['value'] - df_w['p95']).abs()
low_diff = (df_w['value'] - df_w['p5']).abs()
# choose appropriate denominator per row
denom_high = df_w['p95'].abs().clip(lower=eps)
denom_low = df_w['p5'].abs().clip(lower=eps)
df_w['violation_ratio'] = np.where(
    df_w['value'] >= df_w['p95'],
    high_diff / denom_high,
    np.where(df_w['value'] <= df_w['p5'], low_diff / denom_low, 0.0)
)

# Aggregate basic stats per group within window
group_cols = ['cmdb_id', 'kpi_name']
agg_basic = df_w.groupby(group_cols).agg(
    total_points_in_window=('value', 'size'),
    anomaly_count=('is_anomaly', 'sum'),
    max_value=('value', 'max'),
    min_value=('value', 'min'),
    p95=('p95', 'first'),
    p5=('p5', 'first'),
    max_violation_ratio=('violation_ratio', 'max')
).reset_index()

# For earliest anomaly timestamp and up to first 5 anomaly timestamps per group
df_anom = df_w[df_w['is_anomaly']].sort_values(['cmdb_id', 'kpi_name', 'ts_dt'])
# format timestamps as strings in UTC+8
df_anom['ts_str'] = df_anom['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# earliest anomaly timestamp
earliest = df_anom.groupby(group_cols).agg(earliest_anomaly_timestamp=('ts_dt', 'min')).reset_index()
earliest['earliest_anomaly_timestamp'] = earliest['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# first up to 5 anomaly timestamps per group (as comma-separated string)
top_ts = df_anom.groupby(group_cols).agg(anomalous_timestamps=('ts_str', lambda s: ', '.join(s.head(5).tolist()))).reset_index()

# Merge aggregates
agg = agg_basic.merge(earliest, on=group_cols, how='left').merge(top_ts, on=group_cols, how='left')

# Fill NaNs for groups with zero anomalies
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].fillna('')
agg['anomalous_timestamps'] = agg['anomalous_timestamps'].fillna('')

# If there are groups that had no points in window (not present in agg), we might want to include them with zeros.
# Build full list of groups from metric_agg_full and left-join to ensure groups with zero points are considered.
groups_all = metric_agg_full[['cmdb_id', 'kpi_name', 'p95', 'p5', 'total_point_count']].rename(columns={'total_point_count':'total_point_count_global'})
# merge to get groups missing in agg (those had zero points in window)
agg_full = groups_all.merge(agg, on=['cmdb_id','kpi_name','p95','p5'], how='left')

# Fill missing window stats with zeros / empties
agg_full['total_points_in_window'] = agg_full['total_points_in_window'].fillna(0).astype(int)
agg_full['anomaly_count'] = agg_full['anomaly_count'].fillna(0).astype(int)
agg_full['max_value'] = agg_full['max_value'].fillna(np.nan)
agg_full['min_value'] = agg_full['min_value'].fillna(np.nan)
agg_full['max_violation_ratio'] = agg_full['max_violation_ratio'].fillna(0.0)
agg_full['earliest_anomaly_timestamp'] = agg_full['earliest_anomaly_timestamp'].fillna('')
agg_full['anomalous_timestamps'] = agg_full['anomalous_timestamps'].fillna('')

# Select and order columns requested
result_cols = [
    'cmdb_id', 'kpi_name', 'p95', 'p5',
    'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp',
    'max_violation_ratio', 'anomalous_timestamps', 'max_value', 'min_value'
]
agg_result = agg_full[result_cols].copy()

# Sort by anomaly_count desc, then by max_violation_ratio desc
agg_result = agg_result.sort_values(['anomaly_count', 'max_violation_ratio'], ascending=[False, False]).reset_index(drop=True)

# Top 20 groups
anomalies_top20 = agg_result.head(20)

# Keep output compact: show only requested columns (limit anomalous_timestamps length already capped)
anomalies_top20[['cmdb_id','kpi_name','p95','p5','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_violation_ratio','anomalous_timestamps']], anomalies_top20.shape

```
Out[3]:
```
```python
summary = (
    "Anomaly scan summary (global P95/P5 used; window 2022-03-21 10:00:00 to 10:30:00 UTC+8):\n\n"
    "What was done:\n"
    "- For each (cmdb_id, kpi_name) we used the previously computed global P95 and P5 and flagged rows in the incident window as anomalous when value >= P95 or value <= P5.\n\n"
    "Key findings (top groups by anomaly_count):\n"
    "- Many groups have anomaly_count = total_points_in_window (31), with earliest anomaly at 2022-03-21 10:00:00 and anomalies at every minute in the window.\n"
    "- Examples of top affected groups: frontend-2 (container.node-5.container_memory_failures.*), productcatalogservice-0 (container.node-6.container_memory_failures.*), cartservice-1/0/2 (container.* fs_reads / cpu_cfs_throttled metrics), currencyservice2-0, frontend-1, and several node-level metrics (system.io.avg_q_sz, system.process.zombie.num, system.io.r_await).\n"
    "- For many of these groups the stored global P95 and P5 are 0.0, so any positive activity is flagged; this yields extremely large max_violation_ratio values (division by near-zero threshold).\n\n"
    "Implication and next steps:\n"
    "- The anomalies are concentrated and persistent across the entire 10:00–10:30 window for the listed metrics and components; investigate these components (frontend-2, productcatalogservice-0, cartservice pods, the named nodes) and the specific metrics (container memory failures, CPU throttling, fs reads, node I/O and zombie processes).\n"
    "- Because many thresholds are zero, confirm whether those global P95/P5 values are meaningful for these KPIs (e.g., historically zero) or if metrics are sparsely populated; if thresholds are not meaningful, consider alternative baselines before scoring severity.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  p95   p5  total_points_in_window  anomaly_count    earliest_anomaly_timestamp  max_violation_ratio                               anomalous_timestamps
0                frontend-2  container.node-5.container_memory_failures.con...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
1                frontend-2  container.node-5.container_memory_failures.hie...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
2   productcatalogservice-0  container.node-6.container_memory_failures.con...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
3   productcatalogservice-0  container.node-6.container_memory_failures.hie...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
4             cartservice-1       container.node-6.container_fs_reads./dev/vda  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         3.500000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
5             cartservice-0  container.node-6.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
6             cartservice-2  container.node-6.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
7        currencyservice2-0  container.node-6.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
8                frontend-1       container.node-5.container_fs_reads./dev/vda  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
9                    node-1                                 system.io.avg_q_sz  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.400000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
10            cartservice-1  container.node-6.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
11           cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
12                   node-2                          system.process.zombie.num  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
13                   node-4                          system.process.zombie.num  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
14                   node-6                                  system.io.r_await  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
15               frontend-0  container.node-6.container_cpu_cfs_throttled_p...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         5.000000e+08  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
16       currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         1.442220e+08  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
17            cartservice-2  container.node-6.container_cpu_cfs_throttled_s...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         9.060858e+07  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
18            cartservice-0  container.node-6.container_cpu_cfs_throttled_s...  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         8.361402e+07  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
19               frontend-1    container.node-5.container_fs_reads_MB./dev/vda  0.0  0.0                      31             31  2022-03-21 10:00:00 CST+0800         4.947917e+07  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01..., (20, 11))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel: df_w (windowed metric rows with is_anomaly & violation_ratio), agg_result
# Validate existence
try:
    df_w
    agg_result
    tz
except NameError:
    raise RuntimeError("Required variables (df_w, agg_result, tz) not found in kernel.")

# Select top 12 groups by anomaly_count (descending) from agg_result
top12_groups = agg_result.sort_values(['anomaly_count', 'max_violation_ratio'], ascending=[False, False]).head(12).reset_index(drop=True)
top12_keys = top12_groups[['cmdb_id', 'kpi_name']].apply(tuple, axis=1).tolist()

# Prepare containers for faults and group summaries
faults_records = []
group_summaries = []

# Process each group
for cmdb_id, kpi_name in top12_keys:
    df_g = df_w[(df_w['cmdb_id'] == cmdb_id) & (df_w['kpi_name'] == kpi_name)].sort_values('ts_dt').copy()
    df_an = df_g[df_g['is_anomaly']].copy()
    df_an['ts_str'] = df_an['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    if df_an.empty:
        group_summaries.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'total_faults_found': 0,
            'earliest_fault_start': ''
        })
        continue
    # compute diff in seconds between consecutive anomaly rows
    diff_sec = df_an['ts_dt'].diff().dt.total_seconds().fillna(60)
    # new fault when gap != 60 seconds (contiguous per-minute)
    fault_id = (diff_sec != 60).cumsum()
    df_an['fault_id'] = fault_id
    faults = []
    for fid, sub in df_an.groupby('fault_id', sort=True):
        start_ts = sub['ts_dt'].min()
        end_ts = sub['ts_dt'].max()
        length_min = int(sub.shape[0])  # minutes count
        max_val = float(sub['value'].max())
        min_val = float(sub['value'].min())
        max_violation = float(sub['violation_ratio'].max())
        start_str = start_ts.strftime('%Y-%m-%d %H:%M:%S %Z%z')
        end_str = end_ts.strftime('%Y-%m-%d %H:%M:%S %Z%z')
        ts_list = sub['ts_str'].head(5).tolist()
        ts_compact = ', '.join(ts_list)
        faults.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'fault_start_ts': start_str,
            'fault_end_ts': end_str,
            'fault_length_minutes': length_min,
            'fault_max_value': max_val,
            'fault_min_value': min_val,
            'fault_max_violation_ratio': max_violation,
            'anomalous_timestamps_sample': ts_compact
        })
    faults_sorted = sorted(faults, key=lambda x: x['fault_start_ts'])
    faults_limited = faults_sorted[:5]
    for idx, f in enumerate(faults_limited, start=1):
        rec = {
            'cmdb_id': f['cmdb_id'],
            'kpi_name': f['kpi_name'],
            'fault_index': idx,
            'fault_start_timestamp': f['fault_start_ts'],
            'fault_end_timestamp': f['fault_end_ts'],
            'fault_length_minutes': f['fault_length_minutes'],
            'fault_max_value': f['fault_max_value'],
            'fault_min_value': f['fault_min_value'],
            'fault_max_violation_ratio': f['fault_max_violation_ratio'],
            'anomalous_timestamps_sample': f['anomalous_timestamps_sample']
        }
        faults_records.append(rec)
    earliest_start = faults_sorted[0]['fault_start_ts']
    group_max_violation = max(f['fault_max_violation_ratio'] for f in faults)
    group_summaries.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'total_faults_found': len(faults),
        'earliest_fault_start': earliest_start,
        'group_max_violation_ratio': group_max_violation
    })

# Build DataFrames
faults_df = pd.DataFrame(faults_records)
group_summary_df = pd.DataFrame(group_summaries)

# Parse earliest_fault_start without strict format (handles timezone text+offset)
group_summary_df['earliest_dt'] = pd.to_datetime(group_summary_df['earliest_fault_start'], errors='coerce')

# Sort groups by earliest_fault_start (asc) then by group_max_violation_ratio (desc)
group_summary_df = group_summary_df.sort_values(['earliest_dt', 'group_max_violation_ratio'], ascending=[True, False]).reset_index(drop=True)

# Create ordering for groups
group_order = group_summary_df[['cmdb_id','kpi_name']].apply(tuple, axis=1).tolist()
group_rank = {g:i for i,g in enumerate(group_order)}
# attach group_rank to faults_df for sorting
if not faults_df.empty:
    faults_df['group_key'] = faults_df[['cmdb_id','kpi_name']].apply(tuple, axis=1)
    faults_df['group_rank'] = faults_df['group_key'].map(group_rank)
    # sort faults by group_rank then by fault_max_violation_ratio desc
    faults_df = faults_df.sort_values(['group_rank', 'fault_max_violation_ratio'], ascending=[True, False]).reset_index(drop=True)

# Compact outputs
out_cols = [
    'cmdb_id','kpi_name','fault_index','fault_start_timestamp','fault_end_timestamp',
    'fault_length_minutes','fault_max_value','fault_min_value','fault_max_violation_ratio','anomalous_timestamps_sample'
]
faults_df_compact = faults_df[out_cols].copy() if not faults_df.empty else pd.DataFrame(columns=out_cols)
group_summary_compact = group_summary_df[['cmdb_id','kpi_name','total_faults_found','earliest_fault_start','group_max_violation_ratio']].copy()

# Display compact results (variables)
faults_df_compact, group_summary_compact

```
Out[4]:
```
```python
summary = (
    "Fault segmentation summary (window 2022-03-21 10:00:00 to 10:30:00 UTC+8):\n\n"
    "Top 12 (cmdb_id, kpi_name) groups (by anomaly_count) all show a single continuous fault that spans the entire 31-minute window (10:00 → 10:30).\n\n"
    "Affected components and example KPIs (each had a 31-minute contiguous fault starting at 2022-03-21 10:00:00):\n"
    "- frontend-2: container.node-5.container_memory_failures.*\n"
    "- productcatalogservice-0: container.node-6.container_memory_failures.*\n"
    "- cartservice-1 / cartservice-0 / cartservice-2 / cartservice2-0: container fs_reads and cpu_cfs_throttled metrics\n"
    "- currencyservice2-0: container_cpu_cfs_throttled_* metrics\n"
    "- frontend-1: container.node-5.container_fs_reads.*\n"
    "- node-1: system.io.avg_q_sz\n"
    "- node-2 and node-4: system.process.zombie.num (also flagged)\n"
    "- node-6: system.io.r_await\n\n"
    "Key observations:\n"
    "- Each group had anomaly_count == total_points_in_window (31), i.e., anomalies present every minute in the window.\n"
    "- Earliest fault start for all listed groups is 2022-03-21 10:00:00 (UTC+8).\n"
    "- Many global P95/P5 values for these KPIs are 0.0, so any positive measurements are classified as anomalies; this produces extremely large violation ratios (division by near-zero baseline).\n\n"
    "Immediate recommendations:\n"
    "1) Investigate the listed pods and nodes urgently (frontend-2, productcatalogservice-0, cartservice pods, currencyservice2-0, frontend-1 and nodes 1/2/4/6).\n"
    "2) Focus on container memory_failures, CPU throttling, fs reads and node I/O and zombie process metrics—these are the repeatedly flagged KPIs.\n"
    "3) Revisit threshold baselines where P95/P5 == 0.0 to confirm whether zero is an appropriate baseline; if not, use a more robust baseline before computing severity.\n\n"
    "Summary conclusion: persistent, full-window anomalies were detected across multiple pods and nodes centered on container memory failures, CPU throttling, filesystem reads, and node I/O/zombie processes. These should be triaged as correlated infrastructure-level issues starting at 10:00 UTC+8."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  fault_index         fault_start_timestamp           fault_end_timestamp  fault_length_minutes  fault_max_value  fault_min_value  fault_max_violation_ratio                        anomalous_timestamps_sample
0                frontend-2  container.node-5.container_memory_failures.con...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31             33.0              0.0               3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
1                frontend-2  container.node-5.container_memory_failures.hie...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31             33.0              0.0               3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
2   productcatalogservice-0  container.node-6.container_memory_failures.con...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31             33.0              0.0               3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
3   productcatalogservice-0  container.node-6.container_memory_failures.hie...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31             33.0              0.0               3.300000e+10  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
4             cartservice-1       container.node-6.container_fs_reads./dev/vda            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              3.5              0.0               3.500000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
5             cartservice-0  container.node-6.container_cpu_cfs_throttled_p...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              2.0              0.0               2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
6             cartservice-2  container.node-6.container_cpu_cfs_throttled_p...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              2.0              0.0               2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
7        currencyservice2-0  container.node-6.container_cpu_cfs_throttled_p...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              2.0              0.0               2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
8                frontend-1       container.node-5.container_fs_reads./dev/vda            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              2.0              0.0               2.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
9                    node-1                                 system.io.avg_q_sz            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              1.4              0.0               1.400000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
10            cartservice-1  container.node-6.container_cpu_cfs_throttled_p...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              1.0              0.0               1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...
11           cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...            1  2022-03-21 10:00:00 CST+0800  2022-03-21 10:30:00 CST+0800                    31              1.0              0.0               1.000000e+09  2022-03-21 10:00:00 CST+0800, 2022-03-21 10:01...,                     cmdb_id                                           kpi_name  total_faults_found          earliest_fault_start  group_max_violation_ratio
0                frontend-2  container.node-5.container_memory_failures.con...                   1  2022-03-21 10:00:00 CST+0800               3.300000e+10
1                frontend-2  container.node-5.container_memory_failures.hie...                   1  2022-03-21 10:00:00 CST+0800               3.300000e+10
2   productcatalogservice-0  container.node-6.container_memory_failures.con...                   1  2022-03-21 10:00:00 CST+0800               3.300000e+10
3   productcatalogservice-0  container.node-6.container_memory_failures.hie...                   1  2022-03-21 10:00:00 CST+0800               3.300000e+10
4             cartservice-1       container.node-6.container_fs_reads./dev/vda                   1  2022-03-21 10:00:00 CST+0800               3.500000e+09
5             cartservice-0  container.node-6.container_cpu_cfs_throttled_p...                   1  2022-03-21 10:00:00 CST+0800               2.000000e+09
6             cartservice-2  container.node-6.container_cpu_cfs_throttled_p...                   1  2022-03-21 10:00:00 CST+0800               2.000000e+09
7        currencyservice2-0  container.node-6.container_cpu_cfs_throttled_p...                   1  2022-03-21 10:00:00 CST+0800               2.000000e+09
8                frontend-1       container.node-5.container_fs_reads./dev/vda                   1  2022-03-21 10:00:00 CST+0800               2.000000e+09
9                    node-1                                 system.io.avg_q_sz                   1  2022-03-21 10:00:00 CST+0800               1.400000e+09
10            cartservice-1  container.node-6.container_cpu_cfs_throttled_p...                   1  2022-03-21 10:00:00 CST+0800               1.000000e+09
11           cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...                   1  2022-03-21 10:00:00 CST+0800               1.000000e+09)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables if present
try:
    df_trace
    trace_agg_full
    window_start
    window_end
    tz
except NameError:
    # fallback loads (shouldn't be needed in normal flow)
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)
    # compute global p95/p5 for traces if missing (rare fallback)
    trace_agg_full = df_trace.groupby(['cmdb_id','trace_name']).agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05))
    ).reset_index()

# Ensure ts_dt exists and is timezone-aware UTC+8
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter rows inside incident window (inclusive)
df_trace_window = df_trace[df_trace['ts_dt'].between(window_start, window_end)].copy()

# Merge global thresholds (trace_agg_full has cmdb_id, trace_name, p95, p5)
thresholds_trace = trace_agg_full[['cmdb_id','trace_name','p95','p5']].copy()
df_tw = df_trace_window.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies using global thresholds (do not recompute thresholds here)
df_tw['is_anomaly'] = (df_tw['value'] >= df_tw['p95']) | (df_tw['value'] <= df_tw['p5'])

# Prepare formatted timestamp string
df_tw['ts_str'] = df_tw['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Aggregate per (cmdb_id, trace_name)
group_cols = ['cmdb_id','trace_name']
# total points and anomaly counts
agg_basic = df_tw.groupby(group_cols).agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    p95=('p95','first'),
    p5=('p5','first')
).reset_index()

# For earliest/latest anomaly timestamps and samples, operate on anomalous rows
df_trace_anom = df_tw[df_tw['is_anomaly']].copy()

# earliest and latest anomaly per group
if not df_trace_anom.empty:
    times = df_trace_anom.groupby(group_cols).agg(
        earliest_anomaly_timestamp=('ts_dt','min'),
        latest_anomaly_timestamp=('ts_dt','max')
    ).reset_index()
    times['earliest_anomaly_timestamp'] = times['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    times['latest_anomaly_timestamp'] = times['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    # sample up to 5 anomalous (timestamp, value) pairs per group as compact string
    samples = df_trace_anom.sort_values(['cmdb_id','trace_name','ts_dt']).groupby(group_cols).apply(
        lambda d: ', '.join((d['ts_str'] + '|' + d['value'].map(lambda v: ('{:.6g}'.format(v)))) .head(5).tolist())
    ).reset_index().rename(columns={0:'sample_anomalies'})
else:
    times = pd.DataFrame(columns=group_cols+['earliest_anomaly_timestamp','latest_anomaly_timestamp'])
    samples = pd.DataFrame(columns=group_cols+['sample_anomalies'])

# Merge everything
trace_agg_window = agg_basic.merge(times, on=group_cols, how='left').merge(samples, on=group_cols, how='left')

# Fill NAs for groups without anomalies
trace_agg_window['earliest_anomaly_timestamp'] = trace_agg_window['earliest_anomaly_timestamp'].fillna('')
trace_agg_window['latest_anomaly_timestamp'] = trace_agg_window['latest_anomaly_timestamp'].fillna('')
trace_agg_window['sample_anomalies'] = trace_agg_window['sample_anomalies'].fillna('')

# Convert earliest timestamp to datetime for sorting (empty -> NaT)
trace_agg_window['earliest_dt_sort'] = pd.to_datetime(trace_agg_window['earliest_anomaly_timestamp'], errors='coerce')

# Sort by anomaly_count desc, then earliest_anomaly_timestamp asc
trace_agg_window = trace_agg_window.sort_values(['anomaly_count','earliest_dt_sort'], ascending=[False, True]).reset_index(drop=True)

# Select top 20 groups and requested columns
trace_anomaly_summary_top20 = trace_agg_window.head(20)[[
    'cmdb_id','trace_name','p95','p5','total_points_in_window','anomaly_count',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','sample_anomalies'
]].copy()

# Keep output compact: show DataFrame and its shape
trace_anomaly_summary_top20, trace_anomaly_summary_top20.shape

```
Out[5]:
```
```python
summary = (
    "Trace-level anomaly scan (window 2022-03-21 10:00–10:30 UTC+8):\n\n"
    "- Many trace groups were flagged as anomalous for every minute in the window (anomaly_count == total_points_in_window == 30).\n"
    "- Top affected groups are largely adservice-* (trace.from_frontend.* error_rate) and cartservice-* traces (from_checkoutservice, from_frontend, trace.self metrics).\n"
    "- For these groups earliest anomalies start at 2022-03-21 10:00:00 and latest at 2022-03-21 10:29:00 (UTC+8); sample anomalies show repeated values (often 0 for error_rate) across minutes.\n"
    "- Global P95/P5 used for detection are often 0.0 for many trace KPIs, so rows equal to or less than 0 (or equal/above 0) become anomalies — this inflates anomaly counts and can produce misleading results.\n\n"
    "Recommendations:\n"
    "1) Investigate adservice and cartservice trace flows first (error_rate and duration-related traces) since they show persistent anomalies across the full window.\n"
    "2) Re-evaluate global baselines where P95/P5 == 0.0 (determine whether zero is meaningful or reflect sparse/constant data); consider alternative thresholds or smoothing to avoid classifying expected zeros as anomalies.\n"
    "3) Correlate these trace findings with the earlier metric-level anomalies (container memory failures, CPU throttling, fs reads, node I/O) to prioritize infrastructure vs. application causes.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                  trace_name       p95   p5  total_points_in_window  anomaly_count    earliest_anomaly_timestamp      latest_anomaly_timestamp                                   sample_anomalies
0     adservice-0            trace.from_frontend-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
1     adservice-0            trace.from_frontend-1.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
2     adservice-0            trace.from_frontend-2.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
3     adservice-1            trace.from_frontend-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
4     adservice-1            trace.from_frontend-1.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
5     adservice-1            trace.from_frontend-2.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
6     adservice-2            trace.from_frontend-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
7     adservice-2            trace.from_frontend-1.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
8     adservice-2            trace.from_frontend-2.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
9    adservice2-0           trace.from_frontend2-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
10  cartservice-0  trace.from_checkoutservice-0.duration_mean  0.000137  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
11  cartservice-0   trace.from_checkoutservice-0.duration_p95  0.000467  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
12  cartservice-0     trace.from_checkoutservice-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
13  cartservice-0            trace.from_frontend-0.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
14  cartservice-0            trace.from_frontend-1.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
15  cartservice-0            trace.from_frontend-2.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
16  cartservice-0                     trace.self.duration_p95  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
17  cartservice-0                       trace.self.error_rate  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
18  cartservice-1  trace.from_checkoutservice-1.duration_mean  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
19  cartservice-1   trace.from_checkoutservice-1.duration_p95  0.000000  0.0                      30             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:..., (20, 9))```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing kernel variables: df_trace_window, df_trace_anom, group_summary_compact, window_start, window_end, tz
# If any missing, create minimal fallbacks (should not be necessary)
try:
    df_trace_window
    df_trace_anom
    group_summary_compact
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)
    df_trace_window = df_trace[df_trace['ts_dt'].between(window_start, window_end)].copy()
    # Assume no anomalies if trace_agg_full not present
    df_trace_anom = df_trace_window.iloc[0:0].copy()
    group_summary_compact = pd.DataFrame(columns=['cmdb_id'])

# 1) total_trace_series_in_window per cmdb_id
total_series = df_trace_window.groupby('cmdb_id')['trace_name'].nunique().reset_index().rename(columns={'trace_name':'total_trace_series_in_window'})

# 2) anomalous_trace_series_count per cmdb_id (distinct trace_name with >=1 anomaly)
if df_trace_anom.empty:
    anomalous_series = pd.DataFrame(columns=['cmdb_id','anomalous_trace_series_count'])
else:
    anomalous_series = df_trace_anom.groupby('cmdb_id')['trace_name'].nunique().reset_index().rename(columns={'trace_name':'anomalous_trace_series_count'})

# 3 & 4) earliest and latest anomaly timestamps per cmdb_id
if df_trace_anom.empty:
    times = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_timestamp','latest_anomaly_timestamp'])
else:
    times = df_trace_anom.groupby('cmdb_id').agg(
        earliest_anomaly_timestamp=('ts_dt','min'),
        latest_anomaly_timestamp=('ts_dt','max')
    ).reset_index()
    # format as strings in UTC+8
    times['earliest_anomaly_timestamp'] = times['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    times['latest_anomaly_timestamp'] = times['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# 5) sample_trace_names_with_anomalies (up to 10 trace_name values)
if df_trace_anom.empty:
    samples = pd.DataFrame(columns=['cmdb_id','sample_trace_names_with_anomalies'])
else:
    samples = df_trace_anom.groupby('cmdb_id')['trace_name'].apply(lambda arr: ', '.join(pd.unique(arr)[:10])).reset_index().rename(columns={'trace_name':'sample_trace_names_with_anomalies'})

# Merge results into one per-cmdb_id table
merged = total_series.merge(anomalous_series, on='cmdb_id', how='left').merge(times, on='cmdb_id', how='left').merge(samples, on='cmdb_id', how='left')

# Fill NaNs: anomalous count 0, sample empty, timestamps empty
merged['anomalous_trace_series_count'] = merged['anomalous_trace_series_count'].fillna(0).astype(int)
merged['sample_trace_names_with_anomalies'] = merged['sample_trace_names_with_anomalies'].fillna('')
merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_timestamp'].fillna('')
merged['latest_anomaly_timestamp'] = merged['latest_anomaly_timestamp'].fillna('')

# 6) in_metric_faults flag: True if cmdb_id appears in previous metric-faulty components list (top metric-faulty groups)
metric_faulty_cmdbs = set(group_summary_compact['cmdb_id'].dropna().unique())
merged['in_metric_faults'] = merged['cmdb_id'].apply(lambda x: x in metric_faulty_cmdbs)

# For sorting, convert earliest_anomaly_timestamp to datetime (NaT for empty)
merged['earliest_dt_sort'] = pd.to_datetime(merged['earliest_anomaly_timestamp'], errors='coerce')

# Sort by anomalous_trace_series_count desc, then earliest_anomaly_timestamp asc
merged_sorted = merged.sort_values(['anomalous_trace_series_count','earliest_dt_sort'], ascending=[False, True]).reset_index(drop=True)

# Select requested columns and top 20 rows
trace_by_cmdb_top20 = merged_sorted[[
    'cmdb_id','anomalous_trace_series_count','total_trace_series_in_window',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','sample_trace_names_with_anomalies','in_metric_faults'
]].head(20).copy()

# Compact display
trace_by_cmdb_top20, trace_by_cmdb_top20.shape

```
Out[6]:
```
Summary of trace anomalies aggregated by cmdb_id (window 2022-03-21 10:00–10:30 UTC+8):

- Overall pattern:
  - Many components have anomalous trace series across the entire window (earliest = 2022-03-21 10:00:00, latest = 2022-03-21 10:29:00).
  - Several cmdb_ids show anomalies in every trace series present in the window (anomalous_trace_series_count == total_trace_series_in_window).

- Top cmdb_id results (sorted by anomalous_trace_series_count):
  1. checkoutservice-0 / checkoutservice-1 / checkoutservice-2 — anomalous_trace_series_count = 92 of 92 series; earliest 10:00, latest 10:29. (sample traces: trace.from_frontend-*.error_rate, trace.from_frontend-*.row_count, etc.)
  2. frontend-0 / frontend-1 / frontend-2 — anomalous_trace_series_count = 92 of 92 series; earliest 10:00, latest 10:29.
     - frontend-1 and frontend-2 are marked in_metric_faults = True (they also appear in the top metric-faulty components).
  3. productcatalogservice-0 / productcatalogservice-1 / productcatalogservice-2 — anomalous_trace_series_count = 40 of 40 series; earliest 10:00, latest 10:29.
     - productcatalogservice-0 is in_metric_faults = True.
  4. checkoutservice2-0, frontend2-0 — anomalous_trace_series_count = 36 of 36 series.
  5. cartservice-0 / cartservice-1 / cartservice-2 — anomalous_trace_series_count = 28 of 28 series; earliest 10:00, latest 10:29.
     - cartservice-0/1/2 are in_metric_faults = True.
  6. recommendationservice-0/1/2 and various currencyservice instances show smaller totals (24–28 series) but consistent full-window anomalies.

- in_metric_faults correlation:
  - Components that were flagged as metric-faulty in previous metric-level analysis and also show many trace anomalies: frontend-1, frontend-2, productcatalogservice-0, cartservice-0, cartservice-1, cartservice-2 (in_metric_faults = True).
  - Several other cmdb_ids have widespread trace anomalies but were not in the metric-fault list.

- Key takeaway and next steps:
  - There is a broad, persistent set of trace anomalies starting at 10:00 and lasting the full 30-minute window across many services and pods.
  - Prioritize investigation on components that are flagged both by metrics and traces (frontend-1, frontend-2, productcatalogservice-0, cartservice pods) as likely correlated problems.
  - Correlate specific trace types (e.g., trace.from_frontend.* error_rate, trace.from_checkoutservice.* duration/error_rate) with the metric faults (memory failures, CPU throttling, fs reads, node I/O) to determine whether the root cause is infrastructure (nodes/containers) or service-level.
  - Re-evaluate detection baselines where P95/P5 == 0.0 (many trace KPIs had 0.0 thresholds), since that can inflate anomaly counts for series that are normally zero-valued.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  anomalous_trace_series_count  total_trace_series_in_window    earliest_anomaly_timestamp      latest_anomaly_timestamp                  sample_trace_names_with_anomalies  in_metric_faults
0         checkoutservice-0                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.error_rate, trace.from_f...             False
1         checkoutservice-1                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.error_rate, trace.from_f...             False
2         checkoutservice-2                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.error_rate, trace.from_f...             False
3                frontend-0                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_root.error_rate, trace.self.error_r...             False
4                frontend-1                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_root.duration_mean, trace.from_root...              True
5                frontend-2                            92                            92  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_root.error_rate, trace.from_root.ro...              True
6   productcatalogservice-0                            40                            40  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.error_rate, trace...              True
7   productcatalogservice-1                            40                            40  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.error_rate, trace...             False
8   productcatalogservice-2                            40                            40  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.error_rate, trace...             False
9        checkoutservice2-0                            36                            36  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend2-0.error_rate, trace.from_...             False
10              frontend2-0                            36                            36  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_root.duration_p95, trace.from_root....             False
11            cartservice-0                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.duration_mean, tr...              True
12            cartservice-1                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.duration_mean, tr...              True
13            cartservice-2                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.duration_mean, tr...              True
14  recommendationservice-0                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.duration_mean, trace.fro...             False
15  recommendationservice-1                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.error_rate, trace.from_f...             False
16  recommendationservice-2                            28                            28  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_frontend-0.error_rate, trace.from_f...             False
17        currencyservice-0                            24                            24  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.error_rate, trace...             False
18        currencyservice-1                            24                            24  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.duration_p95, tra...             False
19        currencyservice-2                            24                            24  2022-03-21 10:00:00 CST+0800  2022-03-21 10:29:00 CST+0800  trace.from_checkoutservice-0.error_rate, trace...             False, (20, 7))```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables where possible
try:
    df_log
    log_agg_full
    tz
    window_start
    window_end
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_log = pd.read_csv('log.csv')
    df_log['ts_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)
    # fallback: compute global p95/p5 if missing
    log_agg_full = df_log.groupby(['cmdb_id','log_name']).agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05))
    ).reset_index()

# Ensure ts_dt exists and timezone-aware
if 'ts_dt' not in df_log.columns:
    df_log['ts_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define candidates
candidates = ["frontend-1","frontend-2","productcatalogservice-0","cartservice-0","cartservice-1","cartservice-2"]

# Filter log rows in the window
df_log_window = df_log[df_log['ts_dt'].between(window_start, window_end)].copy()

# Merge global thresholds (do NOT recompute thresholds on window)
thresholds_log = log_agg_full[['cmdb_id','log_name','p95','p5']].copy()
df_lw = df_log_window.merge(thresholds_log, on=['cmdb_id','log_name'], how='left')

# Flag anomalies based on global p95/p5
df_lw['is_anomaly'] = (df_lw['value'] >= df_lw['p95']) | (df_lw['value'] <= df_lw['p5'])

# formatted timestamp string
df_lw['ts_str'] = df_lw['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Compute per-(cmdb_id, log_name) aggregates for rows in window (only for candidates)
df_lw_cand = df_lw[df_lw['cmdb_id'].isin(candidates)].copy()

group_cols = ['cmdb_id','log_name']
grouped = df_lw_cand.groupby(group_cols)

# Per-group aggregation
group_agg = grouped.agg(
    total_points_in_window=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    p95=('p95','first'),
    p5=('p5','first')
).reset_index()

# earliest_anomaly_timestamp per group (string)
anom_rows = df_lw_cand[df_lw_cand['is_anomaly']].copy()
if not anom_rows.empty:
    earliest_per_group = anom_rows.groupby(group_cols).agg(earliest_anomaly_ts=('ts_dt','min')).reset_index()
    earliest_per_group['earliest_anomaly_timestamp'] = earliest_per_group['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    earliest_per_group = earliest_per_group.drop(columns=['earliest_anomaly_ts'])
else:
    earliest_per_group = pd.DataFrame(columns=group_cols+['earliest_anomaly_timestamp'])

group_agg = group_agg.merge(earliest_per_group, on=group_cols, how='left')
group_agg['earliest_anomaly_timestamp'] = group_agg['earliest_anomaly_timestamp'].fillna('')

# Store full per-group results for candidates in kernel variable
log_group_agg_candidates = group_agg.copy()

# Now produce per-candidate compact summary rows
summary_records = []
for cmdb in candidates:
    df_c = df_lw_cand[df_lw_cand['cmdb_id']==cmdb].copy()
    total_log_series_in_window = df_c['log_name'].nunique()
    # per-group anomaly counts for this cmdb
    ga = log_group_agg_candidates[log_group_agg_candidates['cmdb_id']==cmdb]
    anomalous_log_series_count = int((ga['anomaly_count'] > 0).sum())
    # earliest and latest anomaly timestamps across logs (from anomalous rows)
    df_c_anom = df_c[df_c['is_anomaly']].copy()
    if not df_c_anom.empty:
        earliest = df_c_anom['ts_dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z%z')
        latest = df_c_anom['ts_dt'].max().strftime('%Y-%m-%d %H:%M:%S %Z%z')
        # sample anomalous rows up to 10 entries
        sample_rows = df_c_anom.sort_values('ts_dt')[['ts_str','log_name','value']].head(10)
        sample_anomalous_rows = sample_rows.apply(lambda r: f"{r['ts_str']}|{r['log_name']}|{('{:.6g}'.format(r['value']))}", axis=1).tolist()
        sample_anomalous_rows = ', '.join(sample_anomalous_rows)
    else:
        earliest = ''
        latest = ''
        sample_anomalous_rows = ''
    # check log.error_count > 0 occurrences
    df_err = df_c[(df_c['log_name']=='log.error_count') & (df_c['value']>0)].copy()
    if not df_err.empty:
        first_error_ts = df_err['ts_dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z%z')
        has_error = True
    else:
        first_error_ts = ''
        has_error = False
    summary_records.append({
        'cmdb_id': cmdb,
        'total_log_series_in_window': int(total_log_series_in_window),
        'anomalous_log_series_count': int(anomalous_log_series_count),
        'earliest_anomaly_timestamp_across_logs': earliest,
        'latest_anomaly_timestamp_across_logs': latest,
        'sample_anomalous_rows': sample_anomalous_rows,
        'log_error_count_gt0': has_error,
        'first_log_error_count_ts': first_error_ts
    })

log_summary_candidates = pd.DataFrame(summary_records)

# Sort summary: anomalous_log_series_count desc, earliest_anomaly_timestamp asc (empty -> later)
log_summary_candidates['earliest_dt_sort'] = pd.to_datetime(log_summary_candidates['earliest_anomaly_timestamp_across_logs'], errors='coerce')
log_summary_candidates = log_summary_candidates.sort_values(['anomalous_log_series_count','earliest_dt_sort'], ascending=[False, True]).reset_index(drop=True)
log_summary_candidates = log_summary_candidates.drop(columns=['earliest_dt_sort'])

# Keep outputs compact: show per-candidate summary (up to 6 rows) and store detailed per-group table
log_summary_candidates, log_group_agg_candidates.head(50)

```
Out[7]:
```
Summary of log analysis for the six candidate components (window 2022-03-21 10:00:00 → 10:30:00 UTC+8):

Common points across candidates
- Each candidate has 2 log series in the window: log.error_count and log.row_count.
- For all six components both log series were marked anomalous (anomalous_log_series_count = 2).
- Earliest anomalies for all candidates appear at 2022-03-21 10:00:00 and latest anomalies are within the window (10:29–10:30).
- No candidate shows log.error_count > 0 in the window (log_error_count_gt0 = False and first_log_error_count_ts empty).
- Many of the log.error_count anomalies stem from global P95/P5 = 0.0 (so zero-valued rows are flagged); treat those with caution because zero baselines can create misleading anomaly flags.

Per-candidate compact rows (one line each)
- frontend-1: total_log_series_in_window=2, anomalous_log_series_count=2, earliest=2022-03-21 10:00:00, latest=2022-03-21 10:30:00. Sample anomalous rows include minute-level log.error_count rows (all 0) and some log.row_count entries (e.g., 10:12:00|log.row_count|...); no log.error_count > 0.
- frontend-2: total=2, anomalous=2, earliest=2022-03-21 10:00:00, latest=2022-03-21 10:29:00. Sample shows log.error_count zeros and log.row_count anomalies; no log.error_count > 0.
- productcatalogservice-0: total=2, anomalous=2, earliest=2022-03-21 10:00:00, latest=10:29:00. Sample shows repeated log.error_count=0 rows and log.row_count anomalies; no log.error_count > 0.
- cartservice-0: total=2, anomalous=2, earliest=2022-03-21 10:00:00, latest=10:29:00. log.error_count rows are all zeros flagged as anomalies; log.row_count shows a few anomalous points (example p95≈885.95, p5≈703.60).
- cartservice-1: total=2, anomalous=2, earliest=2022-03-21 10:00:00, latest=10:29:00. Similar pattern: log.error_count zeros flagged; log.row_count has several anomalous values (p95≈891.10, p5≈722.25).
- cartservice-2: total=2, anomalous=2, earliest=2022-03-21 10:00:00, latest=10:30:00. Same pattern: log.error_count zeros flagged; log.row_count anomalies (p95≈884.00, p5≈696.00).

Actionable notes
- Because log.error_count P95/P5 = 0.0 leads to every zero row being classified as anomalous, treat those anomalies as likely baseline artifacts. Validate whether zero is a meaningful baseline for log.error_count before acting.
- Focus on log.row_count anomalies for signal of traffic/behavior changes (these show non-zero p95/p5 and a few anomalous minutes).
- Correlate these log findings with the metric and trace faults previously identified (frontend, productcatalogservice, cartservice pods) to determine if logs reflect application errors or are secondary to infrastructure issues.

The original code execution output of IPython Kernel is also provided below for reference:

(                   cmdb_id  total_log_series_in_window  anomalous_log_series_count earliest_anomaly_timestamp_across_logs latest_anomaly_timestamp_across_logs                              sample_anomalous_rows  log_error_count_gt0 first_log_error_count_ts
0               frontend-1                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:30:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         
1               frontend-2                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         
2  productcatalogservice-0                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         
3            cartservice-0                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         
4            cartservice-1                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:29:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         
5            cartservice-2                           2                           2           2022-03-21 10:00:00 CST+0800         2022-03-21 10:30:00 CST+0800  2022-03-21 10:00:00 CST+0800|log.error_count|0...                False                         ,                     cmdb_id         log_name  total_points_in_window  anomaly_count      p95       p5    earliest_anomaly_timestamp
0             cartservice-0  log.error_count                      30             30     0.00     0.00  2022-03-21 10:00:00 CST+0800
1             cartservice-0    log.row_count                      30              4   885.95   703.60  2022-03-21 10:00:00 CST+0800
2             cartservice-1  log.error_count                      30             30     0.00     0.00  2022-03-21 10:00:00 CST+0800
3             cartservice-1    log.row_count                      30              4   891.10   722.25  2022-03-21 10:00:00 CST+0800
4             cartservice-2  log.error_count                      31             31     0.00     0.00  2022-03-21 10:00:00 CST+0800
5             cartservice-2    log.row_count                      31              4   884.00   696.00  2022-03-21 10:00:00 CST+0800
6                frontend-1  log.error_count                      31             31     0.00     0.00  2022-03-21 10:00:00 CST+0800
7                frontend-1    log.row_count                      31              4  1616.50  1191.50  2022-03-21 10:12:00 CST+0800
8                frontend-2  log.error_count                      30             30     0.00     0.00  2022-03-21 10:00:00 CST+0800
9                frontend-2    log.row_count                      30              5  1059.20   768.00  2022-03-21 10:01:00 CST+0800
10  productcatalogservice-0  log.error_count                      30             30     0.00     0.00  2022-03-21 10:00:00 CST+0800
11  productcatalogservice-0    log.row_count                      30              4  1008.75   810.25  2022-03-21 10:00:00 CST+0800)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import re

# Reuse existing variables from kernel where available
try:
    df_trace_window
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_trace = pd.read_csv('trace.csv')
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)
    df_trace_window = df_trace[df_trace['ts_dt'].between(window_start, window_end)].copy()

candidates = ["frontend-1","frontend-2","productcatalogservice-0","cartservice-0","cartservice-1","cartservice-2"]

rows = []
for cmdb in candidates:
    df_c = df_trace_window[df_trace_window['cmdb_id'] == cmdb].copy()
    total_distinct = int(df_c['trace_name'].nunique())
    # distinct trace_name counts matching patterns
    distinct_names = df_c['trace_name'].dropna().unique()
    count_from = int(sum(1 for name in distinct_names if name.startswith('trace.from_')))
    count_to = int(sum(1 for name in distinct_names if name.startswith('trace.to_')))
    count_self = int(sum(1 for name in distinct_names if name.startswith('trace.self')))
    # extract service tokens from trace.from_<service> and trace.to_<service>
    from_services = []
    to_services = []
    pattern_from = re.compile(r'^trace\.from_([^\.]+)')
    pattern_to = re.compile(r'^trace\.to_([^\.]+)')
    for name in distinct_names:
        m = pattern_from.match(name)
        if m:
            from_services.append(m.group(1))
        m2 = pattern_to.match(name)
        if m2:
            to_services.append(m2.group(1))
    # unique and limit to 10
    from_services_u = pd.Series(pd.unique(from_services)).dropna().tolist()[:10]
    to_services_u = pd.Series(pd.unique(to_services)).dropna().tolist()[:10]
    sample_from_services = ', '.join(from_services_u)
    sample_to_services = ', '.join(to_services_u)
    # earliest trace timestamp in window for this cmdb_id
    if not df_c.empty:
        earliest_ts = df_c['ts_dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z%z')
    else:
        earliest_ts = ''
    rows.append({
        'cmdb_id': cmdb,
        'total_distinct_trace_names_in_window': total_distinct,
        'count_trace_from_*': count_from,
        'count_trace_to_*': count_to,
        'count_trace_self_*': count_self,
        'sample_from_services': sample_from_services,
        'sample_to_services': sample_to_services,
        'earliest_trace_timestamp_in_window': earliest_ts
    })

trace_candidate_summary = pd.DataFrame(rows)

# Keep output compact: one row per candidate
trace_candidate_summary

```
Out[8]:
```
```python
summary = (
    "Trace-summary for the six candidates (window 2022-03-21 10:00–10:30 UTC+8):\n\n"
    "frontend-1 / frontend-2:\n"
    "- 92 distinct trace names each. Mostly outbound traces (count_trace_to_* = 84), a few inbound (count_trace_from_* = 4) and some self traces (4).\n"
    "- Sample 'to' targets include adservice-0/1/2 and multiple cartservice instances.\n"
    "- Earliest trace in-window: 2022-03-21 10:00:00.\n\n"
    "productcatalogservice-0:\n"
    "- 40 distinct trace names, all are 'trace.from_*' (count_trace_from_* = 40), no 'to' or 'self' traces.\n"
    "- Incoming calls are primarily from checkoutservice pods (e.g., checkoutservice-0/1/2).\n"
    "- Earliest trace in-window: 2022-03-21 10:00:00.\n\n"
    "cartservice-0 / cartservice-1 / cartservice-2:\n"
    "- 28 distinct trace names each. Majority are incoming ('trace.from_*' = 24) and a few self traces (4); no 'trace.to_*'.\n    These incoming traces include calls from checkoutservice pods.\n    Earliest trace in-window: 2022-03-21 10:00:00.\n\n"
    "Overall interpretation:\n"
    "- Tracing topology shows frontends issuing many outgoing calls to adservice and cartservice, while checkoutservice calls are primary callers into productcatalogservice and cartservice.\n"
    "- All six components have trace activity starting at 10:00 and covering the entire incident window; this aligns with metric and trace anomalies found earlier and supports a correlated problem affecting request flows between frontends, checkoutservice, cartservice, and productcatalogservice.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_distinct_trace_names_in_window  count_trace_from_*  count_trace_to_*  count_trace_self_*                               sample_from_services                                 sample_to_services earliest_trace_timestamp_in_window
0               frontend-1                                    92                   4                84                   4                                               root  adservice-0, adservice-1, adservice-2, cartser...       2022-03-21 10:00:00 CST+0800
1               frontend-2                                    92                   4                84                   4                                               root  adservice-0, adservice-1, adservice-2, cartser...       2022-03-21 10:00:00 CST+0800
2  productcatalogservice-0                                    40                  40                 0                   0  checkoutservice-0, checkoutservice-1, checkout...                                                          2022-03-21 10:00:00 CST+0800
3            cartservice-0                                    28                  24                 0                   4  checkoutservice-0, checkoutservice-1, checkout...                                                          2022-03-21 10:00:00 CST+0800
4            cartservice-1                                    28                  24                 0                   4  checkoutservice-0, checkoutservice-1, checkout...                                                          2022-03-21 10:00:00 CST+0800
5            cartservice-2                                    28                  24                 0                   4  checkoutservice-0, checkoutservice-1, checkout...                                                          2022-03-21 10:00:00 CST+0800```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables where possible
try:
    df_metric
    metric_agg_full
    df_trace
    trace_agg_full
    window_start
    window_end
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_agg_full = df_metric.groupby(['cmdb_id','kpi_name']).agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05)),
        total_point_count=('value','size')
    ).reset_index()
    df_trace = pd.read_csv('trace.csv')
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_agg_full = df_trace.groupby(['cmdb_id','trace_name']).agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05)),
        total_point_count=('value','size')
    ).reset_index()
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)

# Ensure timezone-aware ts_dt columns
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter windowed metric/trace rows
df_metric_win = df_metric[df_metric['ts_dt'].between(window_start, window_end)].copy()
df_trace_win = df_trace[df_trace['ts_dt'].between(window_start, window_end)].copy()

# Helper to prepare anomaly marking using global thresholds (merge p95/p5 from global agg)
def mark_anomalies_metric(df_win, agg_global, merge_on=['cmdb_id','kpi_name'], value_col='value'):
    merged = df_win.merge(agg_global[['cmdb_id','kpi_name','p95','p5']], on=merge_on, how='left')
    merged['is_anomaly'] = (merged[value_col] >= merged['p95']) | (merged[value_col] <= merged['p5'])
    merged['ts_str'] = merged['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    return merged

def mark_anomalies_trace(df_win, agg_global, merge_on=['cmdb_id','trace_name'], value_col='value'):
    merged = df_win.merge(agg_global[['cmdb_id','trace_name','p95','p5']], on=merge_on, how='left')
    merged['is_anomaly'] = (merged[value_col] >= merged['p95']) | (merged[value_col] <= merged['p5'])
    merged['ts_str'] = merged['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z')
    return merged

# 1) productcatalogservice-0 metrics: container_memory_failures groups
pc0_metric_rows = df_metric_win[df_metric_win['cmdb_id']=='productcatalogservice-0'].copy()
pc0_mem_rows = pc0_metric_rows[pc0_metric_rows['kpi_name'].str.contains('container_memory_failures', na=False)].copy()

pc0_mem_marked = mark_anomalies_metric(pc0_mem_rows, metric_agg_full)

# Aggregate per kpi_name
if not pc0_mem_marked.empty:
    group_pc_mem = pc0_mem_marked.groupby('kpi_name').agg(
        total_points_in_window=('value','size'),
        anomaly_count=('is_anomaly','sum'),
        p95=('p95','first'),
        p5=('p5','first'),
        earliest_anomaly_ts=('ts_dt', lambda s: s.min() if s.any() else pd.NaT)
    ).reset_index()
    group_pc_mem['earliest_anomaly_timestamp'] = group_pc_mem['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z').fillna('')
    # sample anomalous rows up to 5 per kpi
    anom_pc = pc0_mem_marked[pc0_mem_marked['is_anomaly']].sort_values(['kpi_name','ts_dt'])
    samples_pc = anom_pc.groupby('kpi_name').apply(lambda d: ', '.join((d['ts_str'] + '|' + d['value'].map(lambda v: ('{:.6g}'.format(v)))) .head(5).tolist())).reset_index().rename(columns={0:'sample_anomalous_rows'})
    group_pc_mem = group_pc_mem.merge(samples_pc, on='kpi_name', how='left')
    group_pc_mem['sample_anomalous_rows'] = group_pc_mem['sample_anomalous_rows'].fillna('')
else:
    group_pc_mem = pd.DataFrame(columns=['kpi_name','total_points_in_window','anomaly_count','p95','p5','earliest_anomaly_timestamp','sample_anomalous_rows'])

# Keep compact (<=20 rows)
pc_mem_metrics = group_pc_mem[['kpi_name','total_points_in_window','p95','p5','anomaly_count','earliest_anomaly_timestamp','sample_anomalous_rows']].head(20)

# 2) productcatalogservice-0 traces: all trace_name rows in window (distinct trace_name up to 20)
pc0_trace_rows = df_trace_win[df_trace_win['cmdb_id']=='productcatalogservice-0'].copy()
pc0_trace_marked = mark_anomalies_trace(pc0_trace_rows, trace_agg_full, merge_on=['cmdb_id','trace_name'])

# Aggregate per trace_name
if not pc0_trace_marked.empty:
    g_trace = pc0_trace_marked.groupby('trace_name').agg(
        total_points_in_window=('value','size'),
        anomaly_count=('is_anomaly','sum'),
        p95=('p95','first'),
        p5=('p5','first'),
        earliest_anomaly_ts=('ts_dt', lambda s: s.min() if s.any() else pd.NaT)
    ).reset_index()
    g_trace['earliest_anomaly_timestamp'] = g_trace['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z').fillna('')
    # samples
    anom_trace = pc0_trace_marked[pc0_trace_marked['is_anomaly']].sort_values(['trace_name','ts_dt'])
    samples_trace = anom_trace.groupby('trace_name').apply(lambda d: ', '.join((d['ts_str'] + '|' + d['value'].map(lambda v: ('{:.6g}'.format(v)))) .head(5).tolist())).reset_index().rename(columns={0:'sample_anomalous_rows'})
    g_trace = g_trace.merge(samples_trace, on='trace_name', how='left')
    g_trace['sample_anomalous_rows'] = g_trace['sample_anomalous_rows'].fillna('')
else:
    g_trace = pd.DataFrame(columns=['trace_name','total_points_in_window','p95','p5','anomaly_count','earliest_anomaly_timestamp','sample_anomalous_rows'])

pc_traces = g_trace[['trace_name','total_points_in_window','p95','p5','anomaly_count','earliest_anomaly_timestamp','sample_anomalous_rows']].head(20)

# 3) node-6 metrics: pick requested KPIs and other system.io.* up to 10 groups
node6_rows = df_metric_win[df_metric_win['cmdb_id']=='node-6'].copy()
# Candidate KPI names
requested = ['system.io.r_await','system.io.avg_q_sz','system.process.zombie.num']
# find other system.io.* groups available
sys_io_candidates = sorted([k for k in node6_rows['kpi_name'].unique() if isinstance(k,str) and k.startswith('system.io.')])
# combine and limit to up to 10 unique names, preserving requested first
ordered = []
for r in requested:
    if r in sys_io_candidates and r not in ordered:
        ordered.append(r)
# add other system.io.* until 10
for k in sys_io_candidates:
    if k not in ordered:
        ordered.append(k)
    if len(ordered) >= 10:
        break

node6_selected_rows = node6_rows[node6_rows['kpi_name'].isin(ordered)].copy()
node6_marked = mark_anomalies_metric(node6_selected_rows, metric_agg_full)

# Aggregate per kpi_name
if not node6_marked.empty:
    node6_grp = node6_marked.groupby('kpi_name').agg(
        total_points_in_window=('value','size'),
        anomaly_count=('is_anomaly','sum'),
        p95=('p95','first'),
        p5=('p5','first'),
        earliest_anomaly_ts=('ts_dt', lambda s: s.min() if s.any() else pd.NaT)
    ).reset_index()
    node6_grp['earliest_anomaly_timestamp'] = node6_grp['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z').fillna('')
    anom_node6 = node6_marked[node6_marked['is_anomaly']].sort_values(['kpi_name','ts_dt'])
    samples_node6 = anom_node6.groupby('kpi_name').apply(lambda d: ', '.join((d['ts_str'] + '|' + d['value'].map(lambda v: ('{:.6g}'.format(v)))) .head(5).tolist())).reset_index().rename(columns={0:'sample_anomalous_rows'})
    node6_grp = node6_grp.merge(samples_node6, on='kpi_name', how='left')
    node6_grp['sample_anomalous_rows'] = node6_grp['sample_anomalous_rows'].fillna('')
    # sort by anomaly_count desc and limit to top 10
    node6_grp = node6_grp.sort_values(['anomaly_count','total_points_in_window'], ascending=[False, False]).reset_index(drop=True)
else:
    node6_grp = pd.DataFrame(columns=['kpi_name','total_points_in_window','p95','p5','anomaly_count','earliest_anomaly_timestamp','sample_anomalous_rows'])

node6_metrics = node6_grp[['kpi_name','total_points_in_window','p95','p5','anomaly_count','earliest_anomaly_timestamp','sample_anomalous_rows']].head(10)

# Final compact outputs (three DataFrames)
pc_mem_metrics, pc_traces, node6_metrics

```
Out[9]:
```
Summary (window 2022-03-21 10:00–10:30 UTC+8):

What we found
- productcatalogservice-0 — container.memory_failures metrics
  - All container.node-6.container_memory_failures.* series present for productcatalogservice-0 had total_points_in_window = 31 and anomaly_count = 31, with earliest anomaly at 2022-03-21 10:00:00.
  - Global P95/P5 for these KPIs are 0.0, so every non-zero (or any deviation from zero) or zero-valued point was classified as anomalous. Sample anomalous rows begin at 10:00 and persist every minute across the window.

- productcatalogservice-0 — traces
  - Many trace series present (we returned up to 20). Several trace.error_rate series have p95/p5 = 0.0 and anomaly_count == 30 (anomalies every minute from 10:00).
  - Some duration and row_count traces show smaller but meaningful anomaly counts (e.g., certain duration_mean/p95 groups with 4 anomalies, row_count with 17–22 anomalies).
  - Earliest anomalous trace timestamps are at 2022-03-21 10:00:00.

- node-6 — system IO and related node KPIs
  - Multiple node-6 system.io.* KPIs (system.io.r_await, system.io.avg_q_sz, system.io.w_await, system.io.util, etc.) show high anomaly_count (many 24–31 out of 31 points) and earliest anomalies at 10:00:00.
  - Some node KPIs have non-zero p95 values (e.g., system.io.w_s p95=48.00, system.io.r_s p95=0.50), but many have p5=0.0 which leads to many flagged anomalies as well.

Interpretation and comparison
- Timing: anomalies for productcatalogservice-0 container memory metrics, productcatalogservice-0 traces, and node-6 metrics all start at the same minute (2022-03-21 10:00:00) and persist across the full 30-minute window. That temporal alignment suggests a correlated event affecting both the container and the node at ~10:00.
- Threshold artifacts: many global P95/P5 values are 0.0 (for memory_failures and many trace/error_rate series). When baseline percentiles are zero, normal-zero behaviors or small changes are flagged as anomalies — this inflates counts and violation ratios and reduces confidence in severity scoring for those series.
- Severity signal: node-6 shows multiple system.io KPIs with substantial non-zero p95 values (e.g., IO service time and write throughput related KPIs), while the container.memory_failures KPIs mostly have p95=0.0. If container_memory_failures metrics do contain non-zero values (some samples show positive values), that could indicate real container-level memory failure events; however node IO indicators are also strongly anomalous and provide evidence of node-level I/O pressure.

Conclusion and recommended next steps
- The data does not conclusively single out only the productcatalogservice-0 container or node-6 as the root cause; both show synchronous anomalies beginning at 10:00. This pattern is consistent with either:
  - an underlying node-level problem (disk I/O or resource starvation) that impacts containers running on node-6, or
  - container-level failures that coincide with or trigger node-level symptoms.
- To decide between container vs node root cause, do these targeted checks:
  1) Inspect non-zero, high-confidence metrics (those with non-zero global P95) for node-6 (e.g., system.io.w_s, system.io.r_s, avg_q_sz, w_await) to confirm sustained I/O pressure.
  2) Check container lifecycle events on node-6 around 10:00 (container restarts, OOM Kills, Docker/containerd logs, kubelet events).
  3) Examine kernel logs / dmesg for node-6 (I/O errors, OOM, disk errors) and node-level resource saturation.
  4) Inspect productcatalogservice-0 container logs and process metrics for memory-failure messages, OOM, or JVM/GC errors at 10:00.
  5) Recompute baselines for zero-valued series (or use non-zero robust thresholds) to reduce false-positive anomalies from P95/P5 == 0.0.

Short verdict: both productcatalogservice-0 (container) and node-6 (node) show simultaneous persistent anomalies starting at 10:00. Evidence points to a correlated issue likely involving node-level I/O/resource problems affecting containers on node-6, but you must validate with node logs and container event history to determine the true root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name  total_points_in_window  p95   p5  anomaly_count    earliest_anomaly_timestamp                              sample_anomalous_rows
0  container.node-6.container_memory_failures.con...                      31  0.0  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
1  container.node-6.container_memory_failures.con...                      31  0.0  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
2  container.node-6.container_memory_failures.hie...                      31  0.0  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
3  container.node-6.container_memory_failures.hie...                      31  0.0  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...,                                     trace_name  total_points_in_window         p95          p5  anomaly_count    earliest_anomaly_timestamp                              sample_anomalous_rows
0   trace.from_checkoutservice-0.duration_mean                      30    0.006130    0.004461              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:07:00 CST+0800|0.004457, 2022-03...
1    trace.from_checkoutservice-0.duration_p95                      30    0.006683    0.004676              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:07:00 CST+0800|0.004557, 2022-03...
2      trace.from_checkoutservice-0.error_rate                      30    0.000000    0.000000             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
3       trace.from_checkoutservice-0.row_count                      30    3.000000    1.000000             17  2022-03-21 10:00:00 CST+0800  2022-03-21 10:01:00 CST+0800|1, 2022-03-21 10:...
4   trace.from_checkoutservice-1.duration_mean                      29    0.005791    0.004554              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:09:00 CST+0800|0.004384, 2022-03...
5    trace.from_checkoutservice-1.duration_p95                      29    0.005985    0.004711              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:07:00 CST+0800|0.005996, 2022-03...
6      trace.from_checkoutservice-1.error_rate                      29    0.000000    0.000000             29  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
7       trace.from_checkoutservice-1.row_count                      29    3.000000    1.000000             12  2022-03-21 10:00:00 CST+0800  2022-03-21 10:01:00 CST+0800|1, 2022-03-21 10:...
8   trace.from_checkoutservice-2.duration_mean                      29    0.005680    0.004654              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:08:00 CST+0800|0.005714, 2022-03...
9    trace.from_checkoutservice-2.duration_p95                      29    0.006110    0.004693              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:11:00 CST+0800|0.006259, 2022-03...
10     trace.from_checkoutservice-2.error_rate                      29    0.000000    0.000000             29  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
11      trace.from_checkoutservice-2.row_count                      29    3.000000    1.000000             22  2022-03-21 10:00:00 CST+0800  2022-03-21 10:02:00 CST+0800|3, 2022-03-21 10:...
12         trace.from_frontend-0.duration_mean                      30    0.005096    0.004602              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:06:00 CST+0800|0.017296, 2022-03...
13          trace.from_frontend-0.duration_p95                      30    0.006410    0.005644              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:10:00 CST+0800|0.005634, 2022-03...
14            trace.from_frontend-0.error_rate                      30    0.000000    0.000000             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
15             trace.from_frontend-0.row_count                      30  172.750000  126.350000              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:14:00 CST+0800|125, 2022-03-21 1...
16         trace.from_frontend-1.duration_mean                      30    0.004894    0.004585              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:11:00 CST+0800|0.004565, 2022-03...
17          trace.from_frontend-1.duration_p95                      30    0.006330    0.005662              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:08:00 CST+0800|0.006461, 2022-03...
18            trace.from_frontend-1.error_rate                      30    0.000000    0.000000             30  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
19             trace.from_frontend-1.row_count                      30  166.650000  119.450000              4  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|105, 2022-03-21 1...,              kpi_name  total_points_in_window    p95   p5  anomaly_count    earliest_anomaly_timestamp                              sample_anomalous_rows
0   system.io.r_await                      31   0.00  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
1       system.io.r_s                      31   0.50  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
2     system.io.rkb_s                      31   2.00  0.0             31  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
3   system.io.w_await                      31   0.80  0.0             27  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
4     system.io.await                      31   0.86  0.0             26  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
5       system.io.w_s                      31  48.00  0.0             26  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
6  system.io.avg_q_sz                      31   0.05  0.0             25  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
7     system.io.svctm                      31   0.23  0.0             24  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...
8      system.io.util                      31   1.15  0.0             24  2022-03-21 10:00:00 CST+0800  2022-03-21 10:00:00 CST+0800|0, 2022-03-21 10:...)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: df_metric_win, metric_agg_full, tz, window_start, window_end
# Fallbacks if any missing (should not be necessary)
try:
    df_metric_win
    metric_agg_full
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_start = pd.Timestamp('2022-03-21 10:00:00', tz=tz)
    window_end = pd.Timestamp('2022-03-21 10:30:00', tz=tz)
    df_metric_win = df_metric[df_metric['ts_dt'].between(window_start, window_end)].copy()
    metric_agg_full = df_metric.groupby(['cmdb_id','kpi_name']).agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05))
    ).reset_index()

# Ensure ts_dt exists
if 'ts_dt' not in df_metric_win.columns:
    df_metric_win['ts_dt'] = pd.to_datetime(df_metric_win['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

eps = 1e-6

# Helper to compute group severity table
def severity_table_for(df_win, agg_global, cmdb_filter, kpi_filter_func, limit=None):
    # filter by cmdb_id and kpi predicate
    df_sel = df_win[(df_win['cmdb_id'] == cmdb_filter) & (df_win['kpi_name'].apply(kpi_filter_func))].copy()
    if df_sel.empty:
        return pd.DataFrame(columns=['cmdb_id','kpi_name','total_points_in_window','p95','p5','max_value_in_window','min_value_in_window','earliest_anomaly_timestamp','severity_metric'])
    # merge global thresholds
    df_sel = df_sel.merge(agg_global[['cmdb_id','kpi_name','p95','p5']], on=['cmdb_id','kpi_name'], how='left')
    # mark anomalies using global p95/p5
    df_sel['is_anomaly'] = (df_sel['value'] >= df_sel['p95']) | (df_sel['value'] <= df_sel['p5'])
    # group
    grp = df_sel.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window=('value','size'),
        p95=('p95','first'),
        p5=('p5','first'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        earliest_anom_ts=('ts_dt', lambda s: s[df_sel.loc[s.index,'is_anomaly']].min() if any(df_sel.loc[s.index,'is_anomaly']) else pd.NaT)
    ).reset_index()
    # format earliest timestamp
    grp['earliest_anomaly_timestamp'] = grp['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z%z').fillna('')
    grp = grp.drop(columns=['earliest_anom_ts'])
    # severity metric
    grp['severity_metric'] = (grp['max_value_in_window'] - grp['p95']) / grp['p95'].clip(lower=eps)
    # keep requested columns order
    out_cols = ['cmdb_id','kpi_name','total_points_in_window','p95','p5','max_value_in_window','min_value_in_window','earliest_anomaly_timestamp','severity_metric']
    grp = grp[out_cols]
    # optional limit (after sorting by anomaly_count heuristic not available here) - just limit rows
    if limit is not None:
        grp = grp.head(limit)
    return grp

# 1) productcatalogservice-0 container_memory_failures metrics (up to 20)
pc0_mem_tbl = severity_table_for(
    df_metric_win, metric_agg_full,
    cmdb_filter='productcatalogservice-0',
    kpi_filter_func=lambda k: isinstance(k,str) and 'container_memory_failures' in k,
    limit=20
)

# 2) node-6 selected system.io.* and system.process.zombie.num (choose candidates then pick top 10 by anomaly_count)
# First identify candidate kpis on node-6
node6_all = df_metric_win[df_metric_win['cmdb_id']=='node-6'].copy()
node6_kpis = sorted(node6_all['kpi_name'].dropna().unique().tolist())
# prefer system.io.* and system.process.zombie.num
candidates = [k for k in node6_kpis if isinstance(k,str) and (k.startswith('system.io.') or k=='system.process.zombie.num')]
# If none, fallback to top kpis
if not candidates:
    candidates = node6_kpis[:10]
# Compute anomaly_count per candidate using global thresholds to pick top 10
node6_sel = node6_all[node6_all['kpi_name'].isin(candidates)].merge(metric_agg_full[['cmdb_id','kpi_name','p95','p5']], on=['cmdb_id','kpi_name'], how='left')
node6_sel['is_anomaly'] = (node6_sel['value'] >= node6_sel['p95']) | (node6_sel['value'] <= node6_sel['p5'])
anom_count = node6_sel.groupby('kpi_name')['is_anomaly'].sum().reset_index().rename(columns={'is_anomaly':'anomaly_count'})
# pick top 10 by anomaly_count desc
top_node6_kpis = anom_count.sort_values('anomaly_count', ascending=False).head(10)['kpi_name'].tolist()

# build severity table for node-6 with those top kpis
node6_tbl = severity_table_for(
    df_metric_win, metric_agg_full,
    cmdb_filter='node-6',
    kpi_filter_func=lambda k: k in top_node6_kpis,
    limit=10
)

# Combine and sort by severity_metric desc, limit to 30 rows
combined = pd.concat([pc0_mem_tbl, node6_tbl], ignore_index=True)
combined = combined.sort_values('severity_metric', ascending=False).reset_index(drop=True)
combined_top = combined.head(30)

# Display compact result
combined_top

```
Out[10]:
```
Summary of severity ranking (window 2022-03-21 10:00–10:30 UTC+8):

- Highest reported severity (numeric severity_metric) is for productcatalogservice-0 container_memory_failures groups.
  - Example: container.node-6.container_memory_failures.* shows max_value_in_window = 33, p95 = 0.0 → severity_metric ≈ 3.3e7.
  - Earliest anomaly: 2022-03-21 10:00:00. These series had anomaly_count = 31 (anomalous every minute).

- Node-6 has high-severity IO-related signals as well:
  - system.io.r_await: max=1.00, p95=0.00 → severity_metric ≈ 1.0e6 (earliest 10:00).
  - system.io.rkb_s: max=62.00, p95=2.00 → severity_metric = 30.0 (earliest 10:00).
  - system.io.svctm: severity ≈ 3.35 (p95=0.23, max=1.00).
  - Other system.io.* groups (w_await, w_s, await, util, etc.) show smaller positive severity values; many have earliest anomaly at 10:00 and anomalies across the window.

- Additional observations:
  - Several productcatalogservice-0 memory_failure KPIs also show max=0 and severity_metric=0 (still flagged anomalous due to p95/p5 = 0).
  - system.process.zombie.num on node-6 has p95=325, max=325 → severity 0 (no exceedance), but was anomalous by the earlier boolean test because p5 or equality conditions may apply.
  - system.io.avg_q_sz shows max < p95 producing a negative severity (max 0.04 vs p95 0.05 → severity ≈ -0.2), meaning it never exceeded the p95.

- Important caveat about interpretation:
  - Many global p95 or p5 values are exactly 0.0 for these KPIs. When p95 == 0, the severity metric (max - p95)/p95 becomes extremely large (division by very small baseline) and can massively inflate the numeric severity even for modest absolute values. That makes raw severity_metric unreliable for direct comparison unless p95 > 0 is meaningful for the KPI.
  - Therefore, the very large severity numbers (e.g., 3.3e7 for container_memory_failures and 1.0e6 for r_await) are primarily driven by p95 = 0 and should be treated as a flag to investigate, not a precise measure of impact.

- Practical recommendation:
  1) Investigate productcatalogservice-0 container memory_failures immediately (events start at 10:00 and persist). Check container logs, OOM or memory-failure events, and container restart history on node-6.
  2) Investigate node-6 I/O metrics (rkb_s, r_await, svctm, w_await) for sustained I/O pressure around 10:00. These have non-zero p95 values and show meaningful excursions.
  3) Recompute or choose non-zero, KPI-appropriate baselines for metrics with p95==0 before using severity_metric to prioritize remediation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window     p95     p5  max_value_in_window  min_value_in_window    earliest_anomaly_timestamp  severity_metric
0   productcatalogservice-0  container.node-6.container_memory_failures.con...                      31    0.00    0.0                33.00                  0.0  2022-03-21 10:00:00 CST+0800     3.300000e+07
1   productcatalogservice-0  container.node-6.container_memory_failures.hie...                      31    0.00    0.0                33.00                  0.0  2022-03-21 10:00:00 CST+0800     3.300000e+07
2                    node-6                                  system.io.r_await                      31    0.00    0.0                 1.00                  0.0  2022-03-21 10:00:00 CST+0800     1.000000e+06
3                    node-6                                    system.io.rkb_s                      31    2.00    0.0                62.00                  0.0  2022-03-21 10:00:00 CST+0800     3.000000e+01
4                    node-6                                    system.io.svctm                      31    0.23    0.0                 1.00                  0.0  2022-03-21 10:00:00 CST+0800     3.347826e+00
5                    node-6                                  system.io.w_await                      31    0.80    0.0                 0.87                  0.0  2022-03-21 10:00:00 CST+0800     8.750000e-02
6                    node-6                                      system.io.w_s                      31   48.00    0.0                51.50                  0.0  2022-03-21 10:00:00 CST+0800     7.291667e-02
7                    node-6                                    system.io.await                      31    0.86    0.0                 0.87                  0.0  2022-03-21 10:00:00 CST+0800     1.162791e-02
8   productcatalogservice-0  container.node-6.container_memory_failures.con...                      31    0.00    0.0                 0.00                  0.0  2022-03-21 10:00:00 CST+0800     0.000000e+00
9   productcatalogservice-0  container.node-6.container_memory_failures.hie...                      31    0.00    0.0                 0.00                  0.0  2022-03-21 10:00:00 CST+0800     0.000000e+00
10                   node-6                                     system.io.util                      31    1.15    0.0                 1.15                  0.0  2022-03-21 10:00:00 CST+0800     0.000000e+00
11                   node-6                                      system.io.r_s                      31    0.50    0.0                 0.50                  0.0  2022-03-21 10:00:00 CST+0800     0.000000e+00
12                   node-6                          system.process.zombie.num                      31  325.00  313.0               325.00                325.0  2022-03-21 10:00:00 CST+0800     0.000000e+00
13                   node-6                                 system.io.avg_q_sz                      31    0.05    0.0                 0.04                  0.0  2022-03-21 10:00:00 CST+0800    -2.000000e-01```
```